# 01 — DIFW-KNN: Mixed-Type Distance, Balanced Triplet Loss, & Cost-Sensitive Decision

Notebook integrasi eksperimental untuk seluruh jalur **Proposed DIFW-KNN** pada dataset **UCI Bank Marketing (`data/bank-full.csv`)**:
- **Developer 1 (`src/mixed_type_distance.py`):** *Mixed-Type Distance Metric* ($Q_{05}$–$Q_{95}$ range-normalized numeric + mismatch categorical) dan verifikasi aksioma metrik jarak.
- **Developer 2 (`src/triplet_learning.py` & `src/feature_weight_optimizer.py`):** *Class-Balanced Triplet Sampling*, *Triplet Margin Loss* + *L2 Uniform Prior*, pengecekan gradien analitik vs numerik, *Projected Gradient Descent* pada simpleks probabilitas, dan uji stabilitas multi-seed.
- **Developer 3 (`src/difw_knn.py` & `src/cost_sensitive_decision.py`):** *Local Probability Estimation*, *Cost-Sensitive Bayes Decision Rule* ($\tau^* = C_{\text{FP}} / (C_{\text{FP}} + C_{\text{FN}})$), analisis *trade-off* rasio biaya, serta evaluasi ablasi lengkap pada Hold-out Test Set (`E0`–`E5`).

## 1. Muat Dataset `data/bank-full.csv`, Pembersihan Bebas *Leakage*, & Stratified 80:20 Split

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.edgecolor': '#CCCCCC',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'axes.titleweight': 'bold',
})

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    DEFAULT_RANDOM_STATE,
    DEFAULT_TEST_SIZE,
    clean_deterministic,
    load_raw_dataset,
    stratified_split_data,
)

raw_df = load_raw_dataset(PROJECT_ROOT / 'data' / 'bank-full.csv')
X_clean, y_clean = clean_deterministic(raw_df)
X_train_df, X_test_df, y_train_s, y_test_s = stratified_split_data(
    X_clean, y_clean, test_size=DEFAULT_TEST_SIZE, random_state=DEFAULT_RANDOM_STATE
)
y_train = y_train_s.to_numpy(dtype=np.int64)
y_test = y_test_s.to_numpy(dtype=np.int64)

split_summary = pd.DataFrame([
    {
        'subset': 'Full (Cleaned, duration dropped)',
        'n_rows': len(X_clean),
        'n_cols_raw': X_clean.shape[1],
        'n_no': int((y_clean == 0).sum()),
        'n_yes': int(y_clean.sum()),
        'pct_yes': round(100.0 * float(y_clean.mean()), 4),
    },
    {
        'subset': 'Training Set (80%)',
        'n_rows': len(X_train_df),
        'n_cols_raw': X_train_df.shape[1],
        'n_no': int((y_train == 0).sum()),
        'n_yes': int(y_train.sum()),
        'pct_yes': round(100.0 * float(y_train.mean()), 4),
    },
    {
        'subset': 'Hold-out Test Set (20%)',
        'n_rows': len(X_test_df),
        'n_cols_raw': X_test_df.shape[1],
        'n_no': int((y_test == 0).sum()),
        'n_yes': int(y_test.sum()),
        'pct_yes': round(100.0 * float(y_test.mean()), 4),
    },
])
print(f"Dataset loaded from: {PROJECT_ROOT / 'data' / 'bank-full.csv'}")
print(f"Duration column present? {'duration' in X_clean.columns}")
display(split_summary)

: 

## 2. Developer 1 (`src/mixed_type_distance.py`): *Mixed-Type Distance* & Verifikasi Aksioma Metrik

In [ ]:
from src.mixed_type_distance import MixedTypeDistanceMetric, MixedTypeEncoded

metric = MixedTypeDistanceMetric()
X_train_enc = metric.fit_transform(X_train_df)
X_test_enc = metric.transform(X_test_df)

print(f"Training-only M_pos_train (pdays > -1 median): {metric.feature_engineer.pdays_median_pos_:.2f} hari")
print(f"Encoded features (p = {X_train_enc.n_features}): {list(X_train_enc.feature_names)}")
display(metric.get_numeric_params_table())

# Verifikasi aksioma metrik pada subset acak 200 sampel train vs 200 sampel test
rng = np.random.default_rng(DEFAULT_RANDOM_STATE)
idx_a = rng.choice(X_train_enc.n_samples, size=200, replace=False)
idx_b = rng.choice(X_test_enc.n_samples, size=200, replace=False)
sub_a = MixedTypeEncoded(X_train_enc.numeric[idx_a], X_train_enc.categorical[idx_a], X_train_enc.feature_names)
sub_b = MixedTypeEncoded(X_test_enc.numeric[idx_b], X_test_enc.categorical[idx_b], X_test_enc.feature_names)

d_aa = metric.compute_pairwise_feature_distances(sub_a, sub_a)
d_ab = metric.compute_pairwise_feature_distances(sub_a, sub_b)
d_ba = metric.compute_pairwise_feature_distances(sub_b, sub_a)

self_diag_max = float(np.max(np.abs(np.diagonal(d_aa, axis1=0, axis2=1))))
sym_max_diff = float(np.max(np.abs(d_ab - np.transpose(d_ba, (1, 0, 2)))))
d_min, d_max = float(d_ab.min()), float(d_ab.max())

w_test = np.full(X_train_enc.n_features, 1.0 / X_train_enc.n_features)
dw_batch = metric.compute_weighted_distance_matrix(sub_a, sub_b, weights=w_test, batch_size=32)
dw_exact = np.tensordot(d_ab, w_test, axes=([2], [0]))
batch_max_diff = float(np.max(np.abs(dw_batch - dw_exact)))

axiom_checks = pd.DataFrame([
    {'check': 'Self-distance d_j(x, x) == 0', 'measured': f'{self_diag_max:.2e}', 'status': 'PASS' if self_diag_max == 0.0 else 'FAIL'},
    {'check': 'Symmetry d_j(x, z) == d_j(z, x)', 'measured': f'{sym_max_diff:.2e}', 'status': 'PASS' if sym_max_diff == 0.0 else 'FAIL'},
    {'check': 'Feature distance bounds [0, 1]', 'measured': f'[{d_min:.4f}, {d_max:.4f}]', 'status': 'PASS' if (d_min >= 0.0 and d_max <= 1.0) else 'FAIL'},
    {'check': 'Batched D_w vs exact tensor dot', 'measured': f'{batch_max_diff:.2e}', 'status': 'PASS' if batch_max_diff < 1e-12 else 'FAIL'},
])
display(axiom_checks)

## 3. Developer 2 (`src/triplet_learning.py` & `src/feature_weight_optimizer.py`): *Balanced Triplet Learning*, Gradient Check, & Stabilitas Bobot

In [ ]:
from src.triplet_learning import BalancedTripletSampler, numerical_gradient_check
from src.feature_weight_optimizer import FeatureWeightOptimizer

sampler = BalancedTripletSampler(
    n_triplets=6000,
    positive_anchor_ratio=0.5,
    mining_strategy='semi_hard',
    candidate_pool_size=16,
    margin=0.2,
    random_state=DEFAULT_RANDOM_STATE,
)
triplet_batch = sampler.sample(X_train_enc, y_train, metric)

n_pos_anchors = int((triplet_batch.anchor_labels == 1).sum())
n_neg_anchors = int((triplet_batch.anchor_labels == 0).sum())
same_pos_ok = bool(np.all(y_train[triplet_batch.anchor_idx] == y_train[triplet_batch.positive_idx]))
diff_neg_ok = bool(np.all(y_train[triplet_batch.anchor_idx] != y_train[triplet_batch.negative_idx]))
distinct_ap_ok = bool(np.all(triplet_batch.anchor_idx != triplet_batch.positive_idx))

w_init = np.full(X_train_enc.n_features, 1.0 / X_train_enc.n_features)
grad_check = numerical_gradient_check(
    w_init, triplet_batch.delta_feature_distances, margin=0.2, reg_lambda=0.05, eps=1e-6, tol=1e-4
)

print(f"Triplets sampled    : {triplet_batch.n_triplets:,} (Anchor yes={n_pos_anchors:,}, no={n_neg_anchors:,})")
print(f"Triplet validity    : same_pos={same_pos_ok}, diff_neg={diff_neg_ok}, distinct_ap={distinct_ap_ok}")
print(f"Gradient check      : max_abs_err={grad_check['max_abs_err']:.3e}, max_rel_err={grad_check['max_rel_err']:.3e}, status={'PASS' if grad_check['passed'] else 'FAIL'}")

optimizer = FeatureWeightOptimizer(
    margin=0.2,
    reg_lambda=0.05,
    learning_rate=0.5,
    max_iter=200,
    n_triplets=6000,
    mining_strategy='semi_hard',
    random_state=DEFAULT_RANDOM_STATE,
)
optimizer.fit(X_train_enc, y_train, metric, triplet_batch=triplet_batch)

print(f"Optimization iters  : {optimizer.n_iter_}")
print(f"Triplet Loss        : initial={optimizer.initial_loss_:.6f} -> final={optimizer.final_loss_:.6f}")
print(f"Weight constraints  : min={optimizer.weights_.min():.6f}, sum={optimizer.weights_.sum():.10f}")

weights_df = optimizer.get_weights_table()
display(weights_df.round(6))

In [ ]:
# Uji stabilitas bobot terhadap random seed & visualisasi kurva loss + ranking bobot fitur
stability_df, stability_stats = optimizer.evaluate_seed_stability(
    X_train_enc, y_train, metric, seeds=(42, 100, 2026, 7, 99)
)
print(
    f"Multi-Seed Stability (5 seeds): "
    f"mean_cosine={stability_stats['mean_pairwise_cosine']:.6f}, "
    f"min_cosine={stability_stats['min_pairwise_cosine']:.6f}, "
    f"mean_spearman={stability_stats['mean_pairwise_spearman']:.6f}"
)
display(stability_df.round(6))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0))

iters = np.arange(len(optimizer.loss_history_))
axes[0].plot(iters, optimizer.loss_history_, color='#1f77b4', linewidth=2.2, label='Total Loss $L(w)$')
axes[0].plot(iters, optimizer.hinge_loss_history_, color='#ff7f0e', linestyle='--', linewidth=1.8, label='Triplet Hinge Loss')
axes[0].plot(iters, optimizer.reg_loss_history_, color='#2ca02c', linestyle=':', linewidth=1.8, label='L2 Uniform Prior')
axes[0].set_title('Konvergensi Triplet Margin Loss (Projected GD)')
axes[0].set_xlabel('Iterasi')
axes[0].set_ylabel('Nilai Loss')
axes[0].legend(fontsize=8.5)

w_plot = weights_df.sort_values('weight', ascending=True)
colors = ['#2b5c8f' if w >= w_plot['uniform_weight'].iloc[0] else '#8faadc' for w in w_plot['weight']]
axes[1].barh(w_plot['feature'], w_plot['weight'], color=colors)
axes[1].axvline(w_plot['uniform_weight'].iloc[0], color='#d62728', linestyle='--', linewidth=1.5, label=f"Uniform Prior ({w_plot['uniform_weight'].iloc[0]:.4f})")
axes[1].set_title('Learned Feature Weights (DIFW-KNN)')
axes[1].set_xlabel(r'Bobot Fitur $w_j$ ($\Sigma w_j = 1$)')
axes[1].legend(fontsize=8.5, loc='lower right')

fig.tight_layout()
plt.show()

## 4. Developer 3 (`src/difw_knn.py` & `src/cost_sensitive_decision.py`): *Cost-Sensitive Trade-off* & Ablation Study (`E0`–`E5`) pada Hold-out Test

In [ ]:
from src.difw_knn import evaluate_ablation_suite
from src.cost_sensitive_decision import evaluate_cost_ratio_grid

ablation_df, artifacts = evaluate_ablation_suite(
    X_train_df=X_train_df,
    y_train=y_train,
    X_test_df=X_test_df,
    y_test=y_test,
    k=21,
    c_fp=1.0,
    c_fn=4.0,
    n_triplets=6000,
    margin=0.2,
    reg_lambda=0.05,
    random_state=DEFAULT_RANDOM_STATE,
    batch_size=256,
)

# Gabungkan dengan hasil Baseline KNN (E0 & E1) dari Stage 04 untuk perbandingan lengkap E0-E5
baseline_csv = PROJECT_ROOT / 'outputs' / 'baseline_knn' / 'holdout_test_results.csv'
if baseline_csv.exists():
    base_df = pd.read_csv(baseline_csv)
    base_df = base_df[base_df['model_id'].isin(['E0', 'E1'])].copy()
    base_df['threshold'] = 0.5
    base_df['c_fp'] = 1.0
    base_df['c_fn'] = 4.0
    base_df['total_cost'] = base_df['c_fp'] * base_df['fp'] + base_df['c_fn'] * base_df['fn']
    base_df['mean_cost'] = base_df['total_cost'] / len(y_test)
    cols = [
        'model_id', 'model_name', 'k', 'threshold', 'accuracy', 'precision_yes',
        'recall_yes', 'f1_yes', 'pr_auc', 'roc_auc', 'mcc', 'g_mean',
        'tn', 'fp', 'fn', 'tp', 'total_cost', 'fit_sec', 'predict_sec',
    ]
    full_comp_df = pd.concat([base_df[cols], ablation_df[cols]], ignore_index=True)
else:
    full_comp_df = ablation_df.copy()

print("Perbandingan Lengkap Hold-out Test Set (9,043 observasi, skenario biaya C_FP=1, C_FN=4 -> tau*=0.20):")
display(full_comp_df.round(4))

cost_grid_df = evaluate_cost_ratio_grid(
    y_true=y_test,
    y_prob=artifacts['y_prob_weighted'],
    c_fn_grid=(1.0, 2.0, 3.0, 4.0, 5.0, 7.55, 10.0),
    c_fp=1.0,
)
print("\nTrade-off Rasio Biaya (C_FN / C_FP) pada DIFW-KNN (k=21):")
display(cost_grid_df[['c_fn', 'threshold', 'accuracy', 'precision_yes', 'recall_yes', 'f1_yes', 'g_mean', 'mcc', 'total_cost', 'cost_reduction_pct']].round(4))

In [ ]:
# Visualisasi kurva trade-off Cost-Sensitive & perbandingan E0-E5
fig, axes = plt.subplots(1, 2, figsize=(13.8, 5.0))

axes[0].plot(cost_grid_df['threshold'], cost_grid_df['precision_yes'], marker='o', linewidth=2, label='Precision (yes)')
axes[0].plot(cost_grid_df['threshold'], cost_grid_df['recall_yes'], marker='s', linewidth=2, label='Recall (yes)')
axes[0].plot(cost_grid_df['threshold'], cost_grid_df['f1_yes'], marker='^', linewidth=2, label='F1 (yes)')
axes[0].plot(cost_grid_df['threshold'], cost_grid_df['g_mean'], marker='d', linewidth=2, label='G-Mean')
axes[0].axvline(0.20, color='#d62728', linestyle='--', linewidth=1.4, label=r'$\tau^* = 0.20$ ($C_{FN}=4$)')
axes[0].axvline(0.50, color='#666666', linestyle=':', linewidth=1.4, label=r'Standard $\tau = 0.50$')
axes[0].set_title('Trade-off Threshold Biaya pada DIFW-KNN')
axes[0].set_xlabel(r'Decision Threshold $\tau^* = C_{FP} / (C_{FP} + C_{FN})$')
axes[0].set_ylabel('Skor Metrik')
axes[0].set_ylim(0, 0.85)
axes[0].legend(fontsize=8.2)

x = np.arange(len(full_comp_df))
width = 0.25
axes[1].bar(x - width, full_comp_df['f1_yes'], width=width, color='#4c72b0', label='F1 (yes)')
axes[1].bar(x, full_comp_df['recall_yes'], width=width, color='#55a868', label='Recall (yes)')
axes[1].bar(x + width, full_comp_df['g_mean'], width=width, color='#c44e52', label='G-Mean')
axes[1].set_xticks(x)
axes[1].set_xticklabels(full_comp_df['model_id'], fontweight='bold')
axes[1].set_ylim(0, 0.85)
axes[1].set_title('Perbandingan Ablasi E0–E5 pada Hold-out Test Set')
axes[1].set_ylabel('Skor Metrik')
axes[1].legend(fontsize=8.5)

fig.tight_layout()
plt.show()